# A/B Test - Segment Deep-Dives

**Objective:** Analyze test results across different user segments to find heterogeneous treatment effects.

**Author:** Ashish Patel  
**Date:** 2026-05-26

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('✅ Libraries loaded')

In [ ]:
# Load data
df = pd.read_csv('../data/sample/ab_test_data.csv')
print(f'✅ Loaded {len(df):,} users')

df['user_type'] = df['new_user'].map({0: 'Returning', 1: 'New'})

df.head()

## 2. Segment Analysis Function

In [ ]:
def analyze_segment(data, segment_col):
    """Analyze A/B test results by segment."""
    results = []
    
    for segment in sorted(data[segment_col].unique()):
        seg_data = data[data[segment_col] == segment]
        control = seg_data[seg_data['variant'] == 'control']
        treatment = seg_data[seg_data['variant'] == 'treatment']
        
        n_c, conv_c = len(control), control['converted'].sum()
        n_t, conv_t = len(treatment), treatment['converted'].sum()
        
        rate_c = conv_c / n_c if n_c > 0 else 0
        rate_t = conv_t / n_t if n_t > 0 else 0
        lift = (rate_t / rate_c - 1) * 100 if rate_c > 0 else 0
        
        # Z-test
        if min(n_c, n_t) > 0:
            pooled = (conv_c + conv_t) / (n_c + n_t)
            se = np.sqrt(pooled * (1-pooled) * (1/n_c + 1/n_t)) if pooled > 0 else 1
            z = (rate_t - rate_c) / se if se > 0 else 0
            p = 2 * (1 - stats.norm.cdf(abs(z)))
        else:
            p = 1.0
        
        results.append({
            'Segment': segment,
            'N_Control': n_c,
            'N_Treatment': n_t,
            'Rate_Control': f'{rate_c*100:.2f}%',
            'Rate_Treatment': f'{rate_t*100:.2f}%',
            'Lift': f'{lift:+.1f}%',
            'P_Value': f'{p:.4f}',
            'Significant': '✅' if p < 0.05 else ''
        })
    
    return pd.DataFrame(results)

## 3. Analysis by Device

In [ ]:
device_results = analyze_segment(df, 'device')
print('📱 Results by Device')
print('=' * 80)
print(device_results.to_string(index=False))

In [ ]:
# Visualize by device
fig, ax = plt.subplots(figsize=(10, 5))

devices = df['device'].unique()
x = np.arange(len(devices))
width = 0.35

control_rates = [df[(df['device']==d) & (df['variant']=='control')]['converted'].mean()*100 for d in devices]
treatment_rates = [df[(df['device']==d) & (df['variant']=='treatment')]['converted'].mean()*100 for d in devices]

ax.bar(x - width/2, control_rates, width, label='Control', color='#3498db')
ax.bar(x + width/2, treatment_rates, width, label='Treatment', color='#2ecc71')

ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Conversion Rate by Device', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(devices)
ax.legend()

plt.tight_layout()
plt.savefig('../docs/img/by_device.png', dpi=150)
plt.show()

## 4. Analysis by User Type

In [ ]:
user_results = analyze_segment(df, 'user_type')
print('👤 Results by User Type')
print('=' * 80)
print(user_results.to_string(index=False))

In [ ]:
# Visualize by user type
fig, ax = plt.subplots(figsize=(8, 5))

user_types = df['user_type'].unique()
x = np.arange(len(user_types))
width = 0.35

control_rates = [df[(df['user_type']==u) & (df['variant']=='control')]['converted'].mean()*100 for u in user_types]
treatment_rates = [df[(df['user_type']==u) & (df['variant']=='treatment')]['converted'].mean()*100 for u in user_types]

ax.bar(x - width/2, control_rates, width, label='Control', color='#3498db')
ax.bar(x + width/2, treatment_rates, width, label='Treatment', color='#2ecc71')

ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Conversion Rate by User Type', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(user_types)
ax.legend()

plt.tight_layout()
plt.savefig('../docs/img/by_user_type.png', dpi=150)
plt.show()

## 6. Interaction Effects

In [ ]:
# Device x User Type interaction
print('📊 Device × User Type Interaction')
print('=' * 80)

df['segment'] = df['device'] + '_' + df['user_type']
interaction_results = analyze_segment(df, 'segment')
print(interaction_results.to_string(index=False))

## 7. Multiple Testing Correction

In [ ]:
# Bonferroni correction
n_tests = len(device_results) + len(user_results)
bonferroni_alpha = 0.05 / n_tests

print('⚠️ Multiple Testing Warning')
print('=' * 50)
print(f'Number of segment tests: {n_tests}')
print(f'Original α: 0.05')
print(f'Bonferroni-corrected α: {bonferroni_alpha:.4f}')

# Apply Bonferroni correction
print('\n📊 Significance After Bonferroni Correction:')
print('  Device segments:')
for _, row in device_results.iterrows():
    p = float(row['P_Value'])
    sig = '✅ Significant' if p < bonferroni_alpha else '❌ Not significant'
    print(f'    {row["Segment"]}: p={row["P_Value"]} → {sig}')

print('  User Type segments:')
for _, row in user_results.iterrows():
    p = float(row['P_Value'])
    sig = '✅ Significant' if p < bonferroni_alpha else '❌ Not significant'
    print(f'    {row["Segment"]}: p={row["P_Value"]} → {sig}')

## 8. Summary & Recommendations

In [ ]:
print('=' * 60)
print('📊 SEGMENTATION SUMMARY')
print('=' * 60)
print(f"""
SEGMENT ANALYSIS COMPLETED:
• By Device: {len(device_results)} segments
• By User Type: {len(user_results)} segments  

KEY FINDINGS:
• Check which segments show significant effects
• Look for heterogeneous treatment effects
• Consider if treatment works better for specific groups

RECOMMENDATIONS:
1. If uniform effect → Ship to all users
2. If segment-specific → Consider targeted rollout
3. If negative in some segments → Investigate further

CAUTIONS:
• Apply multiple testing correction
• Segment analysis is exploratory, not confirmatory
• Small segments may have unreliable estimates
""")